# CKB Research Worker R2 — Universal Colab Hot-Swap Worker

One runtime can execute independent Wingless and Yggdrasil bundles in sequence. Each job gets a fresh workspace, Python virtual environment, and Go caches. CKB-plane remains the sole orchestration, qualification, acceptance, promotion, and runtime authority.

Run the setup cell once. Re-run the **RUN NEXT JOB** cell for each new experiment bundle. A job marked `fresh_runtime_required=true` will fail closed if any earlier job has executed in the current Colab runtime.


In [ ]:
import datetime, hashlib, json, os, pathlib, platform, re, shutil, stat, subprocess, sys, zipfile
from google.colab import files

ROOT = pathlib.Path("/content/ckb-colab-r2")
JOBS_ROOT = ROOT / "jobs"
EVIDENCE_ROOT = ROOT / "evidence"
SESSION_PATH = ROOT / "session.json"

ALLOWED_PROJECTS = {"Wingless", "Yggdrasil"}
ALLOWED_EXECUTABLES = {"python", "python3", "go"}
APT_PIN_RE = re.compile(r"^[A-Za-z0-9][A-Za-z0-9+.-]*(?::[A-Za-z0-9]+)?=[A-Za-z0-9][A-Za-z0-9.+:~_-]*$")
WORKER_INFRASTRUCTURE = {
    "python_venv_package": f"python{sys.version_info.major}.{sys.version_info.minor}-venv"
}
SENSITIVE_ENV = {
    "GITHUB_TOKEN","GH_TOKEN","OPENAI_API_KEY","COINBASE_API_KEY","COINBASE_API_SECRET",
    "CKB_OPERATOR_TOKEN","CKB_TOKEN","AWS_ACCESS_KEY_ID","AWS_SECRET_ACCESS_KEY","AZURE_CLIENT_SECRET"
}

ROOT.mkdir(parents=True, exist_ok=True)
EVIDENCE_ROOT.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_extract(zip_path, dest):
    dest=pathlib.Path(dest).resolve()
    dest.mkdir(parents=True, exist_ok=False)
    prefix=str(dest)+os.sep
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            mode=(info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise RuntimeError(f"symlink forbidden: {info.filename}")
            out=(dest / info.filename).resolve()
            if str(out) != str(dest) and not str(out).startswith(prefix):
                raise RuntimeError(f"zip path escape: {info.filename}")
            if info.is_dir():
                out.mkdir(parents=True, exist_ok=True)
                continue
            out.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(out,"xb") as dst:
                shutil.copyfileobj(src,dst)

def load_session():
    if not SESSION_PATH.exists():
        return {"schema":"ckb.colab.session.v1","jobs_executed":0,"projects":[],"last_job":None}
    return json.loads(SESSION_PATH.read_text())

def save_session(session):
    SESSION_PATH.write_text(json.dumps(session,sort_keys=True,separators=(",",":")))

def gpu_present():
    nvidia=shutil.which("nvidia-smi")
    if not nvidia:
        return False
    p=subprocess.run([nvidia,"--query-gpu=name","--format=csv,noheader"],
                     stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=15,shell=False)
    return p.returncode == 0 and bool(p.stdout.strip())

def ensure_venv_support():
    probe=ROOT/"venv-probe"
    if probe.exists():
        shutil.rmtree(probe)
    try:
        p=subprocess.run([sys.executable,"-m","venv",str(probe)],
                         stdout=subprocess.PIPE,stderr=subprocess.PIPE,shell=False)
        if p.returncode == 0:
            shutil.rmtree(probe,ignore_errors=True)
            return
    finally:
        shutil.rmtree(probe,ignore_errors=True)
    if not shutil.which("apt-get"):
        raise RuntimeError("PYTHON_VENV_UNAVAILABLE_AND_APT_GET_MISSING")
    package=WORKER_INFRASTRUCTURE["python_venv_package"]
    print(json.dumps({"worker_infrastructure_dependency_declared":package},sort_keys=True))
    subprocess.run(["apt-get","update","-qq"],check=True,shell=False)
    subprocess.run(["apt-get","install","-y","-qq",package],check=True,shell=False)
    p=subprocess.run([sys.executable,"-m","venv",str(probe)],
                     stdout=subprocess.PIPE,stderr=subprocess.PIPE,shell=False)
    shutil.rmtree(probe,ignore_errors=True)
    if p.returncode != 0:
        raise RuntimeError("PYTHON_VENV_INSTALL_FAILED")

def validate_job(job_root):
    mpath=job_root/"manifest.json"
    if not mpath.is_file():
        raise RuntimeError("manifest.json missing")
    m=json.loads(mpath.read_text())
    assert m["schema"]=="ckb-plane.colab-research-job.v1"
    assert m["project"] in ALLOWED_PROJECTS
    assert re.fullmatch(r"[0-9a-f]{40}",m["source_commit"])
    cmd=m["command"]
    assert isinstance(cmd,list) and 1 <= len(cmd) <= 64
    assert cmd[0].lower() in ALLOWED_EXECUTABLES
    for key,value in m["authority"].items():
        assert value is False, f"authority must be false: {key}"

    deps=m["dependencies"]
    system_packages=deps["system_packages"]
    assert isinstance(system_packages,list) and len(system_packages) <= 64
    assert len(system_packages)==len(set(system_packages))
    for pkg in system_packages:
        assert isinstance(pkg,str) and APT_PIN_RE.fullmatch(pkg), f"system package must be version-pinned: {pkg}"

    pydep=deps["python"]
    req_path=pydep["requirements_lock_path"]
    req_sha=pydep["requirements_lock_sha256"]
    req_actual=job_root/"payload"/"requirements.lock"
    if req_path is None or req_sha is None:
        assert req_path is None and req_sha is None
        assert not req_actual.exists(), "requirements.lock present but not declared"
    else:
        assert req_path=="payload/requirements.lock"
        assert req_actual.is_file()
        assert sha256_file(req_actual)==req_sha

    godep=deps["go"]
    gm_path,gm_sha=godep["go_mod_path"],godep["go_mod_sha256"]
    gs_path,gs_sha=godep["go_sum_path"],godep["go_sum_sha256"]
    go_mod=job_root/"payload"/"go.mod"
    go_sum=job_root/"payload"/"go.sum"
    if gm_path is None or gm_sha is None or gs_path is None or gs_sha is None:
        assert gm_path is None and gm_sha is None and gs_path is None and gs_sha is None
        assert not go_mod.exists() and not go_sum.exists(), "Go dependency files present but not declared"
    else:
        assert gm_path=="payload/go.mod" and gs_path=="payload/go.sum"
        assert go_mod.is_file() and go_sum.is_file()
        assert sha256_file(go_mod)==gm_sha and sha256_file(go_sum)==gs_sha

    control=m["control"]
    for rel,key in [(control["preregistration_path"],"preregistration_sha256"),
                    (control["seed_manifest_path"],"seed_manifest_sha256")]:
        p=(job_root/rel).resolve()
        assert p.is_file()
        assert sha256_file(p)==control[key]
    declared={}
    for f in m["payload_files"]:
        rel=f["path"]
        assert rel.startswith("payload/")
        p=(job_root/rel).resolve()
        assert p.is_file()
        assert sha256_file(p)==f["sha256"] and p.stat().st_size==f["bytes"]
        declared[rel]=f["sha256"]
    payload_root=job_root/"payload"
    actual={f"payload/{p.relative_to(payload_root).as_posix()}"
            for p in payload_root.rglob("*") if p.is_file()}
    expected=set(declared)
    if actual != expected:
        missing=sorted(expected-actual)
        unexpected=sorted(actual-expected)
        raise AssertionError(f"payload file set mismatch; missing={missing}; unexpected={unexpected}")
    return m, sha256_file(mpath)

def install_declared_system_packages(manifest, session):
    packages=list(manifest["dependencies"]["system_packages"])
    if not packages:
        return session
    if int(session.get("jobs_executed",0)) > 0 or session.get("system_packages_installed"):
        raise RuntimeError("DECLARED_SYSTEM_PACKAGES_REQUIRE_FRESH_RUNTIME")
    if not shutil.which("apt-get"):
        raise RuntimeError("DECLARED_SYSTEM_PACKAGES_REQUIRE_APT_GET")
    # Mark the runtime before installation because a partial apt transaction still changes the VM.
    session["system_packages_installed"]=packages
    save_session(session)
    print(json.dumps({"experiment_system_dependencies_declared":packages},sort_keys=True))
    subprocess.run(["apt-get","update","-qq"],check=True,shell=False)
    subprocess.run(["apt-get","install","-y","-qq",*packages],check=True,shell=False)
    return session

def isolated_command_and_env(manifest, work, job_root):
    env=os.environ.copy()
    for key in SENSITIVE_ENV:
        env.pop(key,None)
    env["CKB_COLAB_PROJECT"]=manifest["project"]
    env["CKB_COLAB_EXPERIMENT_ID"]=manifest["experiment_id"]
    env["CKB_COLAB_SOURCE_COMMIT"]=manifest["source_commit"]
    env["PYTHONDONTWRITEBYTECODE"]="1"

    argv=list(manifest["command"])
    exe=argv[0].lower()

    if manifest["accelerator"] == "gpu" and not gpu_present():
        raise RuntimeError("GPU_REQUIRED_BUT_NOT_AVAILABLE")

    if exe in {"python","python3"}:
        venv=job_root/"venv"
        subprocess.run([sys.executable,"-m","venv",str(venv)],check=True,shell=False)
        vpy=venv/("Scripts/python.exe" if os.name=="nt" else "bin/python")
        pydep=manifest["dependencies"]["python"]
        if pydep["requirements_lock_path"] is not None:
            req=work/"requirements.lock"
            lines=[x.strip() for x in req.read_text().splitlines()
                   if x.strip() and not x.lstrip().startswith("#")]
            if not lines or any("--hash=" not in x for x in lines):
                raise RuntimeError("requirements.lock must use pip hashes for every requirement")
            subprocess.run([str(vpy),"-m","pip","install","--require-hashes","-r",str(req)],
                           check=True,shell=False)
        argv[0]=str(vpy)
        env["PIP_NO_INDEX"]="1"
        env["PIP_DISABLE_PIP_VERSION_CHECK"]="1"
        env["PIP_REQUIRE_VIRTUALENV"]="1"
    elif exe=="go":
        go=shutil.which("go")
        if not go:
            raise RuntimeError("GO_TOOLCHAIN_REQUIRED_BUT_NOT_AVAILABLE")
        argv[0]=go
        env["GOPATH"]=str(job_root/"gopath")
        env["GOMODCACHE"]=str(job_root/"gomodcache")
        env["GOCACHE"]=str(job_root/"gocache")
        godep=manifest["dependencies"]["go"]
        if godep["go_mod_path"] is not None:
            subprocess.run([go,"mod","download"],cwd=str(work),env=env,check=True,shell=False)
        env["GOPROXY"]="off"
    env["CKB_DECLARED_DEPENDENCIES_ONLY"]="1"
    return argv,env

def version(cmd):
    try:
        p=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=15,shell=False)
        return p.stdout.decode("utf-8","replace").strip()
    except Exception as e:
        return f"unavailable:{type(e).__name__}"

def collect_outputs(work,evidence_dir):
    collected=[]
    candidates=[]
    for rel in ["classification.json","result.json","results.json","metrics.json","metrics.csv","receipt.json"]:
        p=work/rel
        if p.is_file():
            candidates.append(p)
    results_dir=work/"results"
    if results_dir.is_dir():
        candidates.extend([p for p in results_dir.rglob("*") if p.is_file()])
    for p in sorted(set(candidates)):
        rel=p.relative_to(work)
        out=evidence_dir/"collected"/rel
        out.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(p,out)
        evidence_rel=str(out.relative_to(evidence_dir)).replace(os.sep,"/")
        collected.append({"path":evidence_rel,"sha256":sha256_file(out),"bytes":out.stat().st_size})
    return collected

def run_next_job():
    uploaded=files.upload()
    if len(uploaded)!=1:
        raise RuntimeError("upload exactly one CKB Colab job ZIP")
    _,data=next(iter(uploaded.items()))

    # Hard workspace reset between every job; session/evidence metadata are outside JOBS_ROOT.
    if JOBS_ROOT.exists():
        shutil.rmtree(JOBS_ROOT)
    JOBS_ROOT.mkdir(parents=True,exist_ok=False)

    incoming=JOBS_ROOT/"incoming.zip"
    incoming.write_bytes(data)
    job_extract=JOBS_ROOT/"bundle"
    safe_extract(incoming,job_extract)
    manifest,job_manifest_sha256=validate_job(job_extract)

    session=load_session()
    if session.get("system_packages_installed") and int(session.get("jobs_executed",0))>0:
        raise RuntimeError("RUNTIME_HAS_PERSISTENT_SYSTEM_PACKAGES: disconnect/delete runtime before another job")
    fresh_required=bool(manifest.get("fresh_runtime_required",False))
    if fresh_required and int(session.get("jobs_executed",0))>0:
        raise RuntimeError("FRESH_RUNTIME_REQUIRED: disconnect/delete runtime, reconnect, then run this bundle first")

    for key in SENSITIVE_ENV:
        if os.environ.get(key):
            raise RuntimeError(f"authority credential present in environment: {key}")

    session=install_declared_system_packages(manifest,session)

    work=JOBS_ROOT/"work"
    shutil.copytree(job_extract/"payload",work)
    argv,env=isolated_command_and_env(manifest,work,JOBS_ROOT)

    started=utc_now()
    status="failed"
    exit_code=None
    stdout=b""
    stderr=b""
    try:
        p=subprocess.run(argv,cwd=str(work),env=env,stdout=subprocess.PIPE,stderr=subprocess.PIPE,
                         timeout=int(manifest["budget"]["max_seconds"]),shell=False)
        stdout,stderr=p.stdout,p.stderr
        exit_code=p.returncode
        status="completed" if p.returncode==0 else "failed"
    except subprocess.TimeoutExpired as e:
        status="timed_out"
        stdout=e.stdout or b""
        stderr=e.stderr or b""
    finished=utc_now()

    evdir=JOBS_ROOT/"evidence"
    evdir.mkdir(parents=True,exist_ok=True)
    (evdir/"stdout.log").write_bytes(stdout)
    (evdir/"stderr.log").write_bytes(stderr)
    collected=collect_outputs(work,evdir)

    evidence={
        "schema":"ckb-plane.colab-research-evidence.v1",
        "project":manifest["project"],
        "experiment_id":manifest["experiment_id"],
        "source_commit":manifest["source_commit"],
        "job_manifest_sha256":job_manifest_sha256,
        "status":status,
        "exit_code":exit_code,
        "started_at_utc":started,
        "finished_at_utc":finished,
        "command":manifest["command"],
        "stdout_sha256":sha256_file(evdir/"stdout.log"),
        "stderr_sha256":sha256_file(evdir/"stderr.log"),
        "collected_files":collected,
        "environment":{
            "python":sys.version,
            "platform":platform.platform(),
            "go":version(["go","version"]),
            "gpu":version(["nvidia-smi","--query-gpu=name,driver_version","--format=csv,noheader"]),
            "hot_swap_worker":"r2",
            "jobs_executed_before_this_job":int(session.get("jobs_executed",0)),
            "fresh_runtime_required":fresh_required,
            "python_venv_isolated":manifest["command"][0].lower() in {"python","python3"},
            "go_cache_isolated":manifest["command"][0].lower()=="go",
            "declared_dependencies":manifest["dependencies"],
            "worker_infrastructure":WORKER_INFRASTRUCTURE
        },
        "authority":{
            "ckb_queue_mutated":False,
            "accepted_ref_mutated":False,
            "production_authority_changed":False,
            "broker_accessed":False,
            "repository_write_credentials_present":False,
            "network_isolation_proven":False
        }
    }
    (evdir/"evidence.json").write_text(json.dumps(evidence,sort_keys=True,separators=(",",":")))

    out_zip=EVIDENCE_ROOT/f'{manifest["experiment_id"]}-{job_manifest_sha256[:12]}-evidence.zip'
    with zipfile.ZipFile(out_zip,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(evdir.rglob("*")):
            if p.is_file():
                z.write(p,p.relative_to(evdir))

    session["jobs_executed"]=int(session.get("jobs_executed",0))+1
    projects=list(session.get("projects",[]))
    if manifest["project"] not in projects:
        projects.append(manifest["project"])
    session["projects"]=projects
    session["last_job"]={
        "project":manifest["project"],
        "experiment_id":manifest["experiment_id"],
        "job_manifest_sha256":job_manifest_sha256,
        "finished_at_utc":finished
    }
    save_session(session)

    print(json.dumps({
        "worker":"CKB_COLAB_WORKER_R2",
        "project":manifest["project"],
        "experiment_id":manifest["experiment_id"],
        "status":status,
        "exit_code":exit_code,
        "hot_swap_ready_for_next_job":True,
        "jobs_executed_this_runtime":session["jobs_executed"],
        "evidence_file":out_zip.name
    },indent=2))
    files.download(str(out_zip))
    return evidence

ensure_venv_support()
print("CKB_COLAB_WORKER_R2_READY")
print('Python venv support verified. Run the next cell once per bundle. Wingless and Yggdrasil may be alternated without restarting unless fresh_runtime_required=true.')


## RUN NEXT JOB

Re-run the cell below for every new Wingless or Yggdrasil bundle. The previous job workspace is deleted before the next bundle is unpacked.


In [ ]:
evidence = run_next_job()
